In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

con = duckdb.connect()

historical_data = con.read_parquet(
    str(PROCESSED_DIR / "historical_combined_features_20161130.parquet")
).df()

later_data = con.read_parquet(
    str(PROCESSED_DIR / "later_combined_features_20170228.parquet")
).df()

print("Historical users:", len(historical_data))
print("Later-cohort users:", len(later_data))

Historical users: 882127
Later-cohort users: 970960


## Define Segments Before Examining Churn Rates

Segments use recorded behavior only. Missing logs are not assumed
to prove inactivity, and recorded playback changes do not establish
why a user's behavior changed.

In [2]:
def add_segments(dataset):
    result = dataset.copy()

    result["log_coverage_segment"] = pd.cut(
        result["recorded_days_30d"],
        bins=[-1, 0, 7, 20, 30],
        labels=[
            "No recorded logs",
            "1-7 recorded days",
            "8-20 recorded days",
            "21-30 recorded days",
        ],
    )

    previous_plays = result["recorded_plays_previous_7d"]
    recent_plays = result["recorded_plays_recent_7d"]

    result["recent_to_previous_play_ratio"] = recent_plays.div(
        previous_plays.replace(0, np.nan)
    )

    no_logs = result["has_listening_logs_30d"].eq(0)

    sufficient_baseline = (
        result["recorded_days_previous_7d"].ge(4)
        & previous_plays.ge(20)
    )

    no_recent_logs = result["recorded_days_recent_7d"].eq(0)

    ratio = result["recent_to_previous_play_ratio"]

    result["playback_change_segment"] = np.select(
        [
            no_logs,
            ~sufficient_baseline,
            sufficient_baseline & no_recent_logs,
            sufficient_baseline & ratio.le(0.5),
            sufficient_baseline & ratio.gt(0.5) & ratio.lt(0.8),
        ],
        [
            "No recorded logs",
            "Insufficient previous activity",
            "No recent logs after established activity",
            "Recorded plays declined at least 50%",
            "Recorded plays declined 20-50%",
        ],
        default="Stable or increased recorded plays",
    )

    assert result["log_coverage_segment"].notna().all()
    assert result["playback_change_segment"].notna().all()

    return result

historical_segments = add_segments(historical_data)
later_segments = add_segments(later_data)

print("Behavior segments assigned.")

Behavior segments assigned.


In [3]:
def summarize_segments(dataset, segment_column, cohort):
    overall_rate = dataset["is_churn"].mean()

    summary = (
        dataset.groupby(segment_column, observed=True)
        .agg(
            users=("msno", "size"),
            churned_users=("is_churn", "sum"),
            churn_rate=("is_churn", "mean"),
        )
        .reset_index()
        .rename(columns={segment_column: "segment"})
    )

    summary.insert(0, "cohort", cohort)
    summary["user_share"] = summary["users"] / len(dataset)
    summary["overall_churn_rate"] = overall_rate
    summary["churn_rate_vs_overall"] = (
        summary["churn_rate"] / overall_rate
    )

    assert summary["users"].sum() == len(dataset)
    assert summary["churned_users"].sum() == dataset["is_churn"].sum()

    return summary

coverage_results = pd.concat([
    summarize_segments(
        historical_segments,
        "log_coverage_segment",
        "historical_train",
    ),
    summarize_segments(
        later_segments,
        "log_coverage_segment",
        "later_evaluation",
    ),
], ignore_index=True)

playback_change_results = pd.concat([
    summarize_segments(
        historical_segments,
        "playback_change_segment",
        "historical_train",
    ),
    summarize_segments(
        later_segments,
        "playback_change_segment",
        "later_evaluation",
    ),
], ignore_index=True)

display(coverage_results)
display(playback_change_results)

,cohort,segment,users,churned_users,churn_rate,user_share,overall_churn_rate,churn_rate_vs_overall
0,historical_train,No recorded logs,214650,24179,0.112644,0.243332,0.096627,1.165763
1,historical_train,1-7 recorded days,151420,22440,0.148197,0.171653,0.096627,1.533708
2,historical_train,8-20 recorded days,230590,18578,0.080567,0.261402,0.096627,0.833799
3,historical_train,21-30 recorded days,285467,20040,0.070201,0.323612,0.096627,0.726515
4,later_evaluation,No recorded logs,212019,15713,0.074111,0.218360,0.089942,0.823991
5,later_evaluation,1-7 recorded days,164702,15401,0.093508,0.169628,0.089942,1.039652
6,later_evaluation,8-20 recorded days,287134,27854,0.097007,0.295722,0.089942,1.078551
7,later_evaluation,21-30 recorded days,307105,28362,0.092353,0.316290,0.089942,1.026805


,cohort,segment,users,churned_users,churn_rate,user_share,overall_churn_rate,churn_rate_vs_overall
0,historical_train,Insufficient previous activity,281114,32455,0.115451,0.318677,0.096627,1.194819
1,historical_train,No recent logs after established activity,3787,375,0.099023,0.004293,0.096627,1.024800
2,historical_train,No recorded logs,214650,24179,0.112644,0.243332,0.096627,1.165763
3,historical_train,Recorded plays declined 20-50%,84029,6215,0.073963,0.095257,0.096627,0.765447
4,historical_train,Recorded plays declined at least 50%,68632,5413,0.078870,0.077803,0.096627,0.816233
5,historical_train,Stable or increased recorded plays,229915,16600,0.072201,0.260637,0.096627,0.747212
6,later_evaluation,Insufficient previous activity,315632,29659,0.093967,0.325072,0.089942,1.044752
7,later_evaluation,No recent logs after established activity,5041,700,0.138861,0.005192,0.089942,1.543900
8,later_evaluation,No recorded logs,212019,15713,0.074111,0.218360,0.089942,0.823991
9,later_evaluation,Recorded plays declined 20-50%,97469,9150,0.093876,0.100384,0.089942,1.043740


In [4]:
import json

REPORT_DIR.mkdir(parents=True, exist_ok=True)

coverage_results.to_csv(
    REPORT_DIR / "log_coverage_segments.csv",
    index=False,
)

playback_change_results.to_csv(
    REPORT_DIR / "playback_change_segments.csv",
    index=False,
)

target_segment = "No recent logs after established activity"
reference_segment = "Stable or increased recorded plays"

comparison_rows = []

for cohort in ["historical_train", "later_evaluation"]:
    cohort_results = playback_change_results.loc[
        playback_change_results["cohort"] == cohort
    ]

    target = cohort_results.loc[
        cohort_results["segment"] == target_segment
    ].iloc[0]

    reference = cohort_results.loc[
        cohort_results["segment"] == reference_segment
    ].iloc[0]

    comparison_rows.append({
        "cohort": cohort,
        "target_users": int(target["users"]),
        "target_churn_rate": float(target["churn_rate"]),
        "reference_users": int(reference["users"]),
        "reference_churn_rate": float(reference["churn_rate"]),
        "difference_percentage_points": float(
            100 * (target["churn_rate"] - reference["churn_rate"])
        ),
        "churn_rate_ratio": float(
            target["churn_rate"] / reference["churn_rate"]
        ),
    })

retention_segment_comparison = pd.DataFrame(comparison_rows)

retention_segment_comparison.to_csv(
    REPORT_DIR / "retention_segment_comparison.csv",
    index=False,
)

segment_config = {
    "listening_window_days": 30,
    "recent_window_days": 7,
    "previous_window_days": 7,
    "established_activity_min_previous_days": 4,
    "established_activity_min_previous_plays": 20,
    "strong_decline_ratio_upper_bound": 0.5,
    "moderate_decline_ratio_upper_bound_exclusive": 0.8,
    "target_segment": target_segment,
    "reference_segment": reference_segment,
    "target_rule": (
        "previous recorded days >= 4, previous recorded plays >= 20, "
        "and recent recorded days = 0"
    ),
    "target_selection": "exploratory after examining segment results",
    "intervention_hypothesis": (
        "Test personalized music recommendations or re-engagement reminders."
    ),
    "eligibility_checks": [
        "Verify logging availability.",
        "Verify current subscription status.",
        "Verify contact eligibility.",
    ],
    "causal_effect_established": False,
    "experiment_run": False,
}

(REPORT_DIR / "retention_segment_config.json").write_text(
    json.dumps(segment_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Segment analysis and retention hypothesis saved.")
retention_segment_comparison

Segment analysis and retention hypothesis saved.


,cohort,target_users,target_churn_rate,reference_users,reference_churn_rate,difference_percentage_points,churn_rate_ratio
0,historical_train,3787,0.099023,229915,0.072201,2.682238,1.371498
1,later_evaluation,5041,0.138861,254458,0.093198,4.566325,1.489959


## Segment Insights and Retention Hypothesis

Analyzed churn by recorded listening days and recent playback changes using predefined segment rules.

Low log coverage and playback decline did not consistently identify substantially higher-risk groups across cohorts. Missing logs alone are therefore not treated as proof of disengagement.

A more specific exploratory target consists of users with at least four recorded days and 20 recorded plays in the previous seven-day window, followed by no logs in the most recent seven days.

| Cohort | Target churn rate | Stable/increased-play reference | Difference |
|---|---:|---:|---:|
| Historical training | 9.90% | 7.22% | +2.68 pp |
| Later evaluation | 13.89% | 9.32% | +4.57 pp |

The later target segment contains 5,041 users and has approximately 1.49 times the reference group's churn rate. The direction is consistent across cohorts, but this remains an observational comparison with differing label provenance.

### Retention Hypothesis

After verifying logging availability, subscription status, and contact eligibility, test personalized music recommendations or re-engagement reminders for this segment.

The observed churn-rate difference is not an intervention effect. The reference group is not a randomized control group, and no retention experiment has been run.

See [the segment notebook](notebooks/13_retention_segments.ipynb), [segment comparison](reports/retention_segment_comparison.csv), and [rules and hypothesis](reports/retention_segment_config.json).